In [26]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [27]:
load_dotenv()  # Load environment variables from .env file

model = ChatOpenAI(
    model="gpt-5.4-nano")

In [28]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str
    score: str

In [29]:
def create_outline(state: BlogState) -> BlogState:

    title = state['topic']

    prompt = f"generate a detailed blog outline for the topic: {title}"

    outline = model.invoke(prompt)

    state['outline'] = outline

    return state

In [30]:
def create_blog(state: BlogState) -> BlogState:

    title = state['topic']
    outline = state['outline']

    prompt = f"generate a detailed blog post for the topic: {title} based on the following outline: {outline}"

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [31]:
def score_blog(state: BlogState) -> BlogState:

    content = state['content']

    prompt = f"score the following blog post on a scale of 1-10 based on its quality and relevance: {content}"

    score = model.invoke(prompt).content

    state['score'] = score

    return state

In [33]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('score_blog', score_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'score_blog')
graph.add_edge('score_blog', END)

workflow = graph.compile()


In [34]:
initial_state = {'topic': "The impact of AI on modern education"}
final_state = workflow.invoke(initial_state)

print(final_state['content'])

# The Impact of AI on Modern Education: Promise, Pitfalls, and a Responsible Path Forward

## 1) Introduction: Why AI in Education Matters

Education is shifting fast—from traditional classroom delivery to learning experiences that can adapt to each student. Instead of “one lesson for everyone,” schools are increasingly exploring **data-driven, personalized learning** powered by AI.

**AI in education** generally refers to systems that use machine learning and related technologies to:
- act as **intelligent tutoring systems**
- provide **learning analytics**
- generate or adapt **educational content**
- automate parts of **grading, feedback, and administration**

In this post, you’ll learn:
- how AI changes teaching and learning in real classrooms
- the practical benefits (and the real risks)
- what schools and educators should do next to adopt AI responsibly

---

## 2) The Current State of AI in Education

AI is no longer experimental in many learning environments. Here are common us

In [35]:
print(final_state['score'])

**Score: 8/10**

**Why it scores highly**
- **Relevance:** Extremely on-topic for “AI in education,” covering major areas (tutoring, assessment, analytics, genAI, accessibility, equity, privacy, literacy).
- **Quality/structure:** Clear, logical sectioning with concrete bullet points, risks, and “responsible adoption” guidance throughout.
- **Balanced tone:** Addresses benefits *and* pitfalls (bias, surveillance concerns, misinformation, academic integrity), and emphasizes teacher/human oversight.

**What holds it back**
- **Mostly generic:** It reads like a well-crafted overview rather than adding distinctive insights, research citations, or real case studies.
- **Limited specificity/evidence:** Mentions “run pilots” and “measure outcomes,” but doesn’t provide metrics examples, study references, or implementation details.
- **Audience fit not fully executed:** While it offers an education-focused narrative, it doesn’t deeply tailor to one audience (e.g., school leaders vs. classroom t